# Modelo Baseline: Buy and Hold

Buy and Hold es la predicción más simple posible: el promedio futuro de cada 
activo será igual a su último precio observado en la ventana de entrada. No 
hay entrenamiento, no hay parámetros, no hay aprendizaje.

Formalmente, dado `X` con forma `(n_muestras, ventana_entrada, n_activos)`, 
la predicción es `X[:, -1, :]`, es decir, el último día de cada ventana.


El enunciado pide explícitamente añadir modelos simples como referencia 
. Su utilidad es servir como **cota inferior de honestidad**: cualquier modelo 
neuronal que no supere este MAE no está aprendiendo nada útil de los datos, 
ya que la predicción trivial "mañana = hoy" lo iguala.

## Imports y carga de datos

In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
import warnings
from sklearn.model_selection import train_test_split

warnings.simplefilter(action="ignore", category=FutureWarning)

# Mismos parámetros que el notebook de lectura del profe
start_date = '1945-01-01'
tickers_validos = ['AEP', 'BA', 'CAT', 'CNP', 'CVX', 'DIS', 'DTE', 'ED', 'GD', 
                   'GE', 'HON', 'HPQ', 'IBM', 'IP', 'JNJ', 'KO', 'KR', 'MMM', 
                   'MO', 'MRK', 'MSI', 'PG', 'XOM']

precios_close = yf.download(tickers_validos, start=start_date, 
                            auto_adjust=True, progress=True)['Close']
precios_close.dropna(axis=1, inplace=True)

print(f"Datos cargados: {precios_close.shape}")

[*********************100%***********************]  23 of 23 completed


Datos cargados: (16192, 23)


## Función para crear ventanas

In [2]:
def create_time_series_data(data, input_window_size, output_window_size):
    """
    Genera ventanas X (input_window_size días de entrada) e y (promedio
    de los output_window_size días siguientes) para cada activo.
    """
    X, y = [], []
    data_array = data.values if isinstance(data, pd.DataFrame) else data
    
    for i in range(len(data_array) - input_window_size - output_window_size + 1):
        input_seq = data_array[i : i + input_window_size]
        output_seq = data_array[i + input_window_size : i + input_window_size + output_window_size]
        X.append(input_seq)
        y.append(np.mean(output_seq, axis=0))
    
    return np.array(X), np.array(y)

## Función Buy and Hold + cálculo del MAE

In [3]:
def predict_buy_and_hold(X):
    """
    Predicción: el promedio futuro = último precio observado de la ventana.
    X tiene forma (n_muestras, ventana_entrada, n_activos).
    Devuelve (n_muestras, n_activos).
    """
    return X[:, -1, :]   # último día de cada ventana, para todos los activos


def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

## Loop sobre las 16 combinaciones

In [4]:
input_windows = [5, 10, 30, 90]
output_windows = [1, 5, 30, 90]
RANDOM_SEED = 42

resultados = []

for in_w in input_windows:
    for out_w in output_windows:
        # Generar ventanas
        X, y = create_time_series_data(precios_close, in_w, out_w)
        
        # Split 90/10 (igual que en el notebook del profe)
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.1, shuffle=False, random_state=RANDOM_SEED
        )
        
        # Predicciones Buy and Hold
        y_pred_train = predict_buy_and_hold(X_train)
        y_pred_test = predict_buy_and_hold(X_test)
        
        # MAE
        mae_train = mae(y_train, y_pred_train)
        mae_test = mae(y_test, y_pred_test)
        
        resultados.append({
            'input_window': in_w,
            'output_window': out_w,
            'mae_train': mae_train,
            'mae_test': mae_test
        })

df_resultados = pd.DataFrame(resultados)
df_resultados

,input_window,output_window,mae_train,mae_test
0,5,1,0.187175,1.407602
1,5,5,0.280386,2.142147
2,5,30,0.612016,4.855594
3,5,90,1.037051,8.126336
4,10,1,0.187238,1.407602
5,10,5,0.280511,2.143015
6,10,30,0.612223,4.855594
7,10,90,1.037403,8.126336
8,30,1,0.187560,1.408499
9,30,5,0.280974,2.144562


## Guardar resultados

In [5]:
import os
os.makedirs('../results', exist_ok=True)
df_resultados.to_csv('../results/buy_and_hold_resultados.csv', index=False)
print("Resultados guardados en results/buy_and_hold_resultados.csv")

Resultados guardados en results/buy_and_hold_resultados.csv


## Conclusiones

El baseline Buy and Hold establece la cota de referencia para los 16 pares 
de ventanas (entrada × salida) que pide el enunciado. Las observaciones 
principales:

- **El MAE crece con la ventana de salida.** Pasa de ~1.4 (salida 1 día) a 
  ~8.1 (salida 90 días) en test, lo cual es coherente: predecir el promedio 
  de 90 días futuros es mucho más difícil que predecir 1 solo día. Este 
  patrón marca dónde tendrán más margen de mejora los modelos neuronales.

- **El MAE apenas cambia con la ventana de entrada.** Para una misma ventana 
  de salida, los resultados son prácticamente idénticos entre 5, 10, 30 y 
  90 días de entrada. Es lo esperado: este modelo solo utiliza el último 
  día observado e ignora todo el histórico anterior. Los modelos neuronales 
  sí deberían beneficiarse de ventanas de entrada mayores.

- **El MAE en test es notablemente superior al de train** (por ejemplo, 
  1.04 vs 8.13 en la combinación 90/90). No se trata de overfitting, ya 
  que este modelo no se entrena. La diferencia refleja el cambio de régimen 
  de precios entre el periodo de entrenamiento (datos antiguos, precios 
  más bajos en valor absoluto) y el de test (datos recientes, precios más 
  altos), al hacer un split cronológico sin shuffle. Este efecto justifica 
  la importancia del preprocesado financiero (normalización, log-returns, 
  etc.) que se aplicará en la fase de investigación.

Estos resultados servirán como referencia mínima a batir en los siguientes 
notebooks (regresión lineal, MLP, RNN, CNN y modelos mixtos).